# AAI-540 Machine Learning Operations (MLOps) Final Team Project

- **Name**: Pros Loung
- **Assigment 3.1**: Feature Store - Exercise
- **Professor**: Mark Christenson
- **Course**: Machine Learning Operations (MLOps) (AAI-540-01)

# Instructions
To complete the assignment use Lab 3.1 as an example; build your own feature store and feature groups, and perform some simple feature engineering tasks detailed in the instructions below. 

# Get Started
Complete the lecture presentation and Lab 3.1 tutorial in SageMaker prior to attempting this assignment.

1. To start the assignment, log on to AWS Learner Labs and launch SageMaker Studio.
2. Use the AAI-540 GitHub to clone the lab notebook if you have not already done so.
3. Run the notebook instance and repeat the lab notebook steps for SageMaker feature store using the Homework 3.1 code files and instructions that follow. 
    - / Lab-3-1Links to an external site.
    - / Homework-3-1Links to an external site.

# Data Sets:

Use the /Homework-3-1Links to an external site. housing data and Google Maps data sets.

# Graded Exercise
The housing data set contains information about houses and their values, and the Google Maps raw data set contains information about addresses and their designations. Imagine we are building an ML tool to predict housing prices. To aid with prediction, we want to create a Neighborhood feature group. We can envision this neighborhood feature group helping us predict house prices by giving us a bucket to group new houses into.

## Feature Group:

After completing the above, generate the following features as part of their feature groups and screenshot the following queries against their feature store.

The neighborhood feature group should contain the following features:

- primary_key - neighborhood
    - derived from neighborhood-political
- event_time
    - time of ingestion to the feature store (calculated using Python)
- <1h ocean
    - one hot encoded column derived from ocean_proximity
- inland
    - one hot encoded column derived from ocean_proximity
- island
    - one hot encoded column derived from ocean_proximity
- near bay
    - one hot encoded column derived from ocean_proximity
- near ocean
    - one hot encoded column derived from ocean_proximity
- median house value
    - derived from median_house_value        
    - average this value across all records for a neighborhood        
    - cap this value at 500,000
- median house age        
    - derived from median_house_age        
    - average this value across all records for a neighborhood        
    - discretized by groups of 10 years i.e. 0-9, 10-19, 20-29, etc.
- total households        
    - derived from households        
    - average this value across all records for a neighborhood        
    - must be an integer (round up if needed)
- bedrooms per household        
    - derived from total_bedrooms and households
    - impute missing values by getting average for a postal-code 

# Query the Feature Values:

Please query the feature values from your feature store:

1. ) Brooktree

2. ) Fisherman’s Wharf

3. ) Los Osos

## Setup SageMaker FeatureStore

Let's start by setting up the SageMaker Python SDK and boto client. Note that this notebook requires a `boto3` version above `1.17.21`

In [1]:
import boto3
import sagemaker

original_boto3_version = boto3.__version__
%pip install "boto3>1.17.21"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# from sagemaker.session import Session
from sagemaker.core.helper.session_helper import Session # Require for SageMaker v3 migration

region = boto3.Session().region_name

boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


#### S3 Bucket Setup For The OfflineStore

SageMaker FeatureStore writes the data in the OfflineStore of a FeatureGroup to a S3 bucket owned by you. To be able to write to your S3 bucket, SageMaker FeatureStore assumes an IAM role which has access to it. The role is also owned by you.
Note that the same bucket can be re-used across FeatureGroups. Data in the bucket is partitioned by FeatureGroup.

Set the default s3 bucket name and it will be referenced throughout the notebook.

In [3]:
# You can modify the following to use a bucket of your choosing
default_s3_bucket_name = feature_store_session.default_bucket()
prefix = "sagemaker-featurestore-homwork3.1" # modify the bucket to sagemaker-featurestore-homwork3.1

print(default_s3_bucket_name)

sagemaker-us-east-1-944202758041


Set up the IAM role. This role gives SageMaker FeatureStore access to your S3 bucket. 

<div class="alert alert-block alert-warning">
<b>Note:</b> In this example we use the default SageMaker role, assuming it has both <b>AmazonSageMakerFullAccess</b> and <b>AmazonSageMakerFeatureStoreAccess</b> managed policies. If not, please make sure to attach them to the role before proceeding.
</div>

In [4]:
# from sagemaker import get_execution_role
from sagemaker.core.helper.session_helper import get_execution_role # Update for SageMaker 3

# You can modify the following to use a role of your choosing. See the documentation for how to create this.
role = get_execution_role()
print(role)

arn:aws:iam::944202758041:role/LabRole


# Load the datasets

In [4]:
# Grab path to the dataset.csv

from pathlib import Path

current_directory = Path.cwd()

print("Current directory:")
print(current_directory)

print("\nCSV files found:")
csv_files = list(current_directory.rglob("*.csv"))

if not csv_files:
    print("No CSV files found.")
else:
    for file_path in csv_files:
        print(file_path.resolve())

Current directory:
/home/sagemaker-user/aai-540-homework/homework-3-1

CSV files found:
/home/sagemaker-user/aai-540-homework/homework-3-1/housing.csv
/home/sagemaker-user/aai-540-homework/homework-3-1/housing_gmaps_data_raw.csv


In [9]:
import pandas as pd
from pathlib import Path

housing_path = Path("/home/sagemaker-user/aai-540-homework/homework-3-1/housing.csv")
gmaps_path = Path("/home/sagemaker-user/aai-540-homework/homework-3-1/housing_gmaps_data_raw.csv")

for p in (housing_path, gmaps_path):
    if not p.is_file():
        raise FileNotFoundError(f"File not found: {p}")

df_housing = pd.read_csv(housing_path)
df_gmaps = pd.read_csv(gmaps_path)

# Exploratory Data Analysis

In [22]:
# Housing Dataset
print("Housing Dataset Information:")
print("Rows, Columns:", df_housing.shape)
print(df_housing.info())
print("First 5 row:")
df_housing.head()


Housing Dataset Information:
Rows, Columns: (20640, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB
None
First 5 row:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [23]:
# Google Maps Dataset
print("\nGoogle Maps Dataset Information:")
print("Rows, Columns:", df_gmaps.shape)
print(df_gmaps.info())
print("First 5 row:")
df_gmaps.head()


Google Maps Dataset Information:
Rows, Columns: (12590, 30)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12590 entries, 0 to 12589
Data columns (total 30 columns):
 #   Column                                                                              Non-Null Count  Dtype  
---  ------                                                                              --------------  -----  
 0   street_number                                                                       11188 non-null  object 
 1   route                                                                               12210 non-null  object 
 2   locality-political                                                                  12403 non-null  object 
 3   administrative_area_level_2-political                                               12543 non-null  object 
 4   administrative_area_level_1-political                                               12587 non-null  object 
 5   country-political                 

,street_number,route,locality-political,administrative_area_level_2-political,administrative_area_level_1-political,country-political,postal_code,address,longitude,latitude,...,establishment-natural_feature,airport-establishment-point_of_interest,political-sublocality-sublocality_level_1,administrative_area_level_3-political,post_box,establishment-light_rail_station-point_of_interest-transit_station,establishment-point_of_interest,aquarium-establishment-park-point_of_interest-tourist_attraction-zoo,campground-establishment-lodging-park-point_of_interest-rv_park-tourist_attraction,cemetery-establishment-park-point_of_interest
0,3130,Grizzly Peak Boulevard,Berkeley,Alameda County,California,United States,94705.0,"3130 Grizzly Peak Blvd, Berkeley, CA 94705, USA",-122.23,37.88,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2005,Tunnel Road,Oakland,Alameda County,California,United States,94611.0,"2005 Tunnel Rd, Oakland, CA 94611, USA",-122.22,37.86,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6886,Chabot Road,Oakland,Alameda County,California,United States,94618.0,"6886 Chabot Rd, Oakland, CA 94618, USA",-122.24,37.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6365,Florio Street,Oakland,Alameda County,California,United States,94618.0,"6365 Florio St, Oakland, CA 94618, USA",-122.25,37.85,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5407,Bryant Avenue,Oakland,Alameda County,California,United States,94618.0,"5407 Bryant Ave, Oakland, CA 94618, USA",-122.25,37.84,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Inspect Dataset

## Inspect Dataset

The provided dataset is a synthetic dataset with two tables: identity and transactions. They can both be joined by the `TransactionId` column. The transaction table contains information about a particular transaction such as amount, credit or debit card while the identity table contains information about the user such as device type and browser. The transaction must exist in the transaction table, but might not always be available in the identity table.

The objective of the model is to predict if a transaction is fraudulent or not, given the transaction record.

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io

s3_client = boto3.client("s3", region_name=region)

fraud_detection_bucket_name = f"sagemaker-example-files-prod-{region}" # S3 Sagement Public database
identity_file_key = (
    "datasets/tabular/fraud_detection/synthethic_fraud_detection_SA/sampled_identity.csv"
)
transaction_file_key = (
    "datasets/tabular/fraud_detection/synthethic_fraud_detection_SA/sampled_transactions.csv"
)

# Use s3_client to get the datasets
identity_data_object = s3_client.get_object(
    Bucket=fraud_detection_bucket_name, Key=identity_file_key
)
transaction_data_object = s3_client.get_object(
    Bucket=fraud_detection_bucket_name, Key=transaction_file_key
)

# Convert dataset stored in binary to pandas frame
identity_data = pd.read_csv(io.BytesIO(identity_data_object["Body"].read()))
transaction_data = pd.read_csv(io.BytesIO(transaction_data_object["Body"].read()))



In [20]:
identity_data.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20
0,2990130,-5,38780.0,0.0,0.0,0.0,-70,0,1,100.0,...,32,80,253,241,260,125,T,F,F,T
1,2990266,-10,69246.0,0.0,0.0,0.0,-67,0,2,100.0,...,47,47,122,33,38,60,T,F,T,F
2,2992553,-45,348819.0,NaN,NaN,0.0,-73,0,0,100.0,...,21,143,268,111,2,135,F,F,T,F
3,2994568,-15,337170.0,NaN,NaN,0.0,-10,1,2,100.0,...,55,127,253,202,135,49,F,F,T,T
4,2994749,-5,680670.0,NaN,NaN,8.0,-1,2,2,100.0,...,52,43,257,7,19,254,F,F,T,T


In [21]:
transaction_data.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,card1,card2,card3,card4,card5,card6,...,F17,N1,N2,N3,N4,N5,N6,N7,N8,N9
0,3343087,0,8810855,29.00,12469,360.0,150.0,mastercard,126.0,debit,...,519,F,F,T,T,T,T,T,F,T
1,3307318,0,7955295,107.95,16188,178.0,150.0,mastercard,224.0,debit,...,773,F,T,T,T,F,F,F,F,T
2,3555327,0,15084339,159.95,1825,555.0,150.0,visa,226.0,debit,...,771,F,T,F,F,T,T,T,T,F
3,3310736,0,8017157,159.95,10057,225.0,150.0,mastercard,224.0,debit,...,903,T,T,F,T,T,F,T,F,F
4,3034711,0,1127470,117.00,11444,555.0,150.0,visa,226.0,debit,...,579,T,T,T,F,T,F,T,F,F


# Feature Engineering

In [22]:
# Feature Engineering

# Round the data to 5-decimal point
identity_data = identity_data.round(5)
transaction_data = transaction_data.round(5)

# Fill the null data with 0
identity_data = identity_data.fillna(0)
transaction_data = transaction_data.fillna(0)

# Feature transformations for this dataset are applied before ingestion into FeatureStore.
# One hot encode card4, card6
encoded_card_bank = pd.get_dummies(transaction_data["card4"], prefix="card_bank")
encoded_card_type = pd.get_dummies(transaction_data["card6"], prefix="card_type")

transformed_transaction_data = pd.concat(
    [transaction_data, encoded_card_type, encoded_card_bank], axis=1
)
# blank space is not allowed in feature name
transformed_transaction_data = transformed_transaction_data.rename(
    columns={"card_bank_american express": "card_bank_american_express"}
)

In [23]:
identity_data.head()

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20
0,2990130,-5,38780.0,0.0,0.0,0.0,-70,0,1,100.0,...,32,80,253,241,260,125,T,F,F,T
1,2990266,-10,69246.0,0.0,0.0,0.0,-67,0,2,100.0,...,47,47,122,33,38,60,T,F,T,F
2,2992553,-45,348819.0,0.0,0.0,0.0,-73,0,0,100.0,...,21,143,268,111,2,135,F,F,T,F
3,2994568,-15,337170.0,0.0,0.0,0.0,-10,1,2,100.0,...,55,127,253,202,135,49,F,F,T,T
4,2994749,-5,680670.0,0.0,0.0,8.0,-1,2,2,100.0,...,52,43,257,7,19,254,F,F,T,T


In [24]:
transformed_transaction_data.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,card1,card2,card3,card4,card5,card6,...,N8,N9,card_type_0,card_type_credit,card_type_debit,card_bank_0,card_bank_american_express,card_bank_discover,card_bank_mastercard,card_bank_visa
0,3343087,0,8810855,29.00,12469,360.0,150.0,mastercard,126.0,debit,...,F,T,False,False,True,False,False,False,True,False
1,3307318,0,7955295,107.95,16188,178.0,150.0,mastercard,224.0,debit,...,F,T,False,False,True,False,False,False,True,False
2,3555327,0,15084339,159.95,1825,555.0,150.0,visa,226.0,debit,...,T,F,False,False,True,False,False,False,False,True
3,3310736,0,8017157,159.95,10057,225.0,150.0,mastercard,224.0,debit,...,F,F,False,False,True,False,False,False,True,False
4,3034711,0,1127470,117.00,11444,555.0,150.0,visa,226.0,debit,...,F,F,False,False,True,False,False,False,False,True


## Ingest Data into FeatureStore


In this step we will create the FeatureGroups representing the transaction and identity tables.

#### Define FeatureGroups

In [25]:
from time import gmtime, strftime, sleep

# append time to feature group to avoid overwriting when run notebook again. For production, you should
# dave only development and production feature groups, not feature groups that are new everytime when notebook run.
identity_feature_group_name = "identity-feature-group-" + strftime("%d-%H-%M-%S", gmtime())
transaction_feature_group_name = "transaction-feature-group-" + strftime("%d-%H-%M-%S", gmtime())

In [26]:
from sagemaker.core.resources import FeatureGroup

from sagemaker.core.shapes import (
    FeatureDefinition,
    OnlineStoreConfig,
    OfflineStoreConfig,
    S3StorageConfig,
)


# In SageMaker v3, FeatureGroup.create() both defines and creates the resource;# identity_feature_group/transaction_feature_group are assigned once created below.

In [27]:
import time

current_time_sec = int(round(time.time()))


def cast_object_to_string(data_frame):
    for label in data_frame.columns:
        if data_frame.dtypes[label] == "object":
            data_frame[label] = data_frame[label].astype("str").astype("string")


# cast object dtype to string. The SageMaker FeatureStore Python SDK will then map the string dtype to String feature type.
cast_object_to_string(identity_data)
cast_object_to_string(transformed_transaction_data)

# record identifier and event time feature names
record_identifier_feature_name = "TransactionID"
event_time_feature_name = "EventTime"

# append EventTime feature
identity_data[event_time_feature_name] = pd.Series(
    [current_time_sec] * len(identity_data), dtype="float64"
)
transformed_transaction_data[event_time_feature_name] = pd.Series(
    [current_time_sec] * len(transaction_data), dtype="float64"
)

# v3 has no auto-detect helper, so map pandas dtypes to FeatureDefinitions manually.
dtype_to_feature_type = {
    "object": "String",
    "string": "String",
    "int64": "Integral",
    "int32": "Integral",
    "float64": "Fractional",
    "float32": "Fractional",
}


def dataframe_to_feature_definitions(data_frame):
    return [
        FeatureDefinition(
            feature_name=column,
            feature_type=dtype_to_feature_type.get(str(data_frame[column].dtype), "String"),
        )
        for column in data_frame.columns
    ]


identity_feature_definitions = dataframe_to_feature_definitions(identity_data)
transaction_feature_definitions = dataframe_to_feature_definitions(transformed_transaction_data)

# Verify the inferred schema before it's passed to FeatureGroup.create().
print("Identity feature definitions:")
for feature in identity_feature_definitions:
    print(f"  {feature.feature_name}: {feature.feature_type}")

print("\nTransaction feature definitions:")
for feature in transaction_feature_definitions:
    print(f"  {feature.feature_name}: {feature.feature_type}")

Identity feature definitions:
  TransactionID: Integral
  id_01: Integral
  id_02: Fractional
  id_03: Fractional
  id_04: Fractional
  id_05: Fractional
  id_06: Integral
  id_07: Integral
  id_08: Integral
  id_09: Fractional
  id_10: Integral
  id_11: Integral
  id_12: Integral
  id_13: Integral
  id_14: Integral
  id_15: Integral
  id_16: Integral
  id_17: String
  id_18: String
  id_19: String
  id_20: String
  EventTime: Fractional

Transaction feature definitions:
  TransactionID: Integral
  isFraud: Integral
  TransactionDT: Integral
  TransactionAmt: Fractional
  card1: Integral
  card2: Fractional
  card3: Fractional
  card4: String
  card5: Fractional
  card6: String
  B1: Integral
  B2: Integral
  B3: Integral
  B4: Integral
  B5: Integral
  B6: Integral
  B7: Integral
  B8: Integral
  B9: Integral
  B10: Integral
  B11: Integral
  B12: Integral
  F1: Integral
  F2: Integral
  F3: Integral
  F4: Integral
  F5: Integral
  F6: Integral
  F7: Integral
  F8: Integral
  F9: Inte

#### Create FeatureGroups in SageMaker FeatureStore

In [28]:
offline_store_config = OfflineStoreConfig(
    s3_storage_config=S3StorageConfig(s3_uri=f"s3://{default_s3_bucket_name}/{prefix}")
)

identity_feature_group = FeatureGroup.create(
    feature_group_name=identity_feature_group_name,
    record_identifier_feature_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    feature_definitions=identity_feature_definitions,
    role_arn=role,
    online_store_config=OnlineStoreConfig(enable_online_store=True),
    offline_store_config=offline_store_config,
    session=boto_session,
)

transaction_feature_group = FeatureGroup.create(
    feature_group_name=transaction_feature_group_name,
    record_identifier_feature_name=record_identifier_feature_name,
    event_time_feature_name=event_time_feature_name,
    feature_definitions=transaction_feature_definitions,
    role_arn=role,
    online_store_config=OnlineStoreConfig(enable_online_store=True),
    offline_store_config=offline_store_config,
    session=boto_session,
)

identity_feature_group.wait_for_status("Created")
transaction_feature_group.wait_for_status("Created")
print("FeatureGroups successfully created.")

[09/19/26 12:43:10] INFO     Creating feature_group resource.                                    resources.py:11769

                    WARNING  No region provided. Using default region.                                 utils.py:361

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:5                                                                                    │
│                                                                                                  │
│    2 │   s3_storage_config=S3StorageConfig(s3_uri=f"s3://{default_s3_bucket_name}/{prefix}")     │
│    3 )                                                                                           │
│    4                                                                                             │
│ ❱  5 identity_feature_group = FeatureGroup.create(                                               │
│    6 │   feature_group_name=identity_feature_group_name,                                         │
│    7 │   record_identifier_feature_name=record_identifier_feature_name,                          │
│    8 │   event_time_feature_name=event_time_feature_name,                                        │
│                                                                                                  │
│ c:\Users\Loung\USD\AAI-540 MLOps\.aai540\Lib\site-packages\sagemaker\core\resources.py:11705 in  │
│ wrapper                                                                                          │
│                                                                                                  │
│   11702 │   │   │   │   },                                                                       │
│   11703 │   │   │   │   "role_arn": {"type": "string"},                                          │
│   11704 │   │   │   }                                                                            │
│ ❱ 11705 │   │   │   return create_func(                                                          │
│   11706 │   │   │   │   *args,                                                                   │
│   11707 │   │   │   │   **Base.get_updated_kwargs_with_configured_attributes(                    │
│   11708 │   │   │   │   │   config_schema_for_resource, "FeatureGroup", **kwargs                 │
│                                                                                                  │
│ c:\Users\Loung\USD\AAI-540 MLOps\.aai540\Lib\site-packages\sagemaker\core\resources.py:142 in    │
│ wrapper                                                                                          │
│                                                                                                  │
│     139 │   │   @functools.wraps(func)                                                           │
│     140 │   │   def wrapper(*args, **kwargs):                                                    │
│     141 │   │   │   config = dict(arbitrary_types_allowed=True)                                  │
│ ❱   142 │   │   │   return validate_call(config=config)(func)(*args, **kwargs)                   │
│     143 │   │                                                                                    │
│     144 │   │   return wrapper                                                                   │
│     145                                                                                          │
│                                                                                                  │
│ c:\Users\Loung\USD\AAI-540                                                                       │
│ MLOps\.aai540\Lib\site-packages\pydantic\_internal\_validate_call.py:40 in wrapper_function      │
│                                                                                                  │
│    37 │   │                                                                                      │
│    38 │   │   @functools.wraps(wrapped)                                                          │
│    39 │   │   def wrapper_function(*args, **kwargs):                                             │
│ ❱  40 │   │   │   return wrapper(*args, **kwargs)          

Confirm the FeatureGroup has been created by using the DescribeFeatureGroup and ListFeatureGroups APIs.

In [ ]:
identity_feature_group.refresh()

In [ ]:
transaction_feature_group.refresh

In [ ]:
sagemaker_client.list_feature_groups()  # use boto client to list FeatureGroups

#### PutRecords into FeatureGroup

After the FeatureGroups have been created, we can put data into the FeatureGroups by using the PutRecord API. This API can handle high TPS and is designed to be called by different streams. The data from all of these Put requests is buffered and written to S3 in chunks. The files will be written to the offline store within a few minutes of ingestion. For this example, to accelerate the ingestion process, we are specifying multiple workers to do the job simultaneously. It will take ~1min to ingest data to the 2 FeatureGroups, respectively.

In [ ]:
identity_feature_group.ingest(data_frame=identity_data, max_workers=3, wait=True)

In [ ]:
transaction_feature_group.ingest(data_frame=transformed_transaction_data, max_workers=5, wait=True)

To confirm that data has been ingested, we can quickly retrieve a record from the online store:

In [ ]:
record_identifier_value = str(2990130)

featurestore_runtime.get_record(
    FeatureGroupName=transaction_feature_group_name,
    RecordIdentifierValueAsString=record_identifier_value,
)

We can also retrieve a record of each feature group from the online store:

In [ ]:
featurestore_runtime.batch_get_record(
    Identifiers=[
        {
            "FeatureGroupName": identity_feature_group_name,
            "RecordIdentifiersValueAsString": ["2990130"],
        },
        {
            "FeatureGroupName": transaction_feature_group_name,
            "RecordIdentifiersValueAsString": ["2990130"],
        },
    ]
)

The SageMaker Python SDK’s FeatureStore class also provides the functionality to generate Hive DDL commands. Schema of the table is generated based on the feature definitions. Columns are named after feature name and data-type are inferred based on feature type.

In [ ]:
print(identity_feature_group.as_hive_ddl())

In [ ]:
print(transaction_feature_group.as_hive_ddl())

Now let's wait for the data to appear in our offline store before moving forward to creating a dataset. This will take approximately 5 minutes.

In [ ]:
account_id = boto3.client("sts").get_caller_identity()["Account"]
print(account_id)

identity_feature_group_resolved_output_s3_uri = (
    identity_feature_group.describe()
    .get("OfflineStoreConfig")
    .get("S3StorageConfig")
    .get("ResolvedOutputS3Uri")
)
transaction_feature_group_resolved_output_s3_uri = (
    transaction_feature_group.describe()
    .get("OfflineStoreConfig")
    .get("S3StorageConfig")
    .get("ResolvedOutputS3Uri")
)

identity_feature_group_s3_prefix = identity_feature_group_resolved_output_s3_uri.replace(
    f"s3://{default_s3_bucket_name}/", ""
)
transaction_feature_group_s3_prefix = transaction_feature_group_resolved_output_s3_uri.replace(
    f"s3://{default_s3_bucket_name}/", ""
)

offline_store_contents = None
while offline_store_contents is None:
    objects_in_bucket = s3_client.list_objects(
        Bucket=default_s3_bucket_name, Prefix=transaction_feature_group_s3_prefix
    )
    if "Contents" in objects_in_bucket and len(objects_in_bucket["Contents"]) > 1:
        offline_store_contents = objects_in_bucket["Contents"]
    else:
        print("Waiting for data in offline store...\n")
        sleep(60)

print("Data available.")

SageMaker FeatureStore adds metadata for each record that's ingested into the offline store.

## Build Training Dataset

SageMaker FeatureStore automatically builds the Glue Data Catalog for FeatureGroups (you can optionally turn it on/off while creating the FeatureGroup). In this example, we want to create one training dataset with FeatureValues from both identity and transaction FeatureGroups. This is done by utilizing the auto-built Catalog. We run an Athena query that joins the data stored in the offline store in S3 from the 2 FeatureGroups. 

In [ ]:
identity_query = identity_feature_group.athena_query()
transaction_query = transaction_feature_group.athena_query()

identity_table = identity_query.table_name
transaction_table = transaction_query.table_name

query_string = (
    'SELECT * FROM "'
    + transaction_table
    + '" LEFT JOIN "'
    + identity_table
    + '" ON "'
    + transaction_table
    + '".transactionid = "'
    + identity_table
    + '".transactionid'
)
print("Running " + query_string)

# run Athena query. The output is loaded to a Pandas dataframe.
# dataset = pd.DataFrame()
identity_query.run(
    query_string=query_string,
    output_location="s3://" + default_s3_bucket_name + "/" + prefix + "/query_results/",
)
identity_query.wait()
dataset = identity_query.as_dataframe()

dataset

In [ ]:
# Prepare query results for training.
query_execution = identity_query.get_query_execution()
query_result = (
    "s3://"
    + default_s3_bucket_name
    + "/"
    + prefix
    + "/query_results/"
    + query_execution["QueryExecution"]["QueryExecutionId"]
    + ".csv"
)
print(query_result)

# Select useful columns for training with target column as the first.
dataset = dataset[
    [
        "isfraud",
        "transactiondt",
        "transactionamt",
        "card1",
        "card2",
        "card3",
        "card5",
        "card_type_credit",
        "card_type_debit",
        "card_bank_american_express",
        "card_bank_discover",
        "card_bank_mastercard",
        "card_bank_visa",
        "id_01",
        "id_02",
        "id_03",
        "id_04",
        "id_05",
    ]
]

# Write to csv in S3 without headers and index column.
dataset.to_csv("dataset.csv", header=False, index=False)
s3_client.upload_file("dataset.csv", default_s3_bucket_name, prefix + "/training_input/dataset.csv")
dataset_uri_prefix = "s3://" + default_s3_bucket_name + "/" + prefix + "/training_input/"

dataset

## Train and Deploy the Model

Now it's time to launch a Training job to fit our model. We use the gradient boosting algorithm provided by XGBoost libary to fit our data. Call the SageMaker XGBoost container and construct a generic SageMaker estimator.

In [ ]:
training_image = sagemaker.image_uris.retrieve("xgboost", region, "1.0-1")

#### Construct a SageMaker generic estimator using the SageMaker XGBoost container

In [ ]:
training_output_path = "s3://" + default_s3_bucket_name + "/" + prefix + "/training_output"

from sagemaker.estimator import Estimator

training_model = Estimator(
    training_image,
    role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size=5,
    max_run=3600,
    input_mode="File",
    output_path=training_output_path,
    sagemaker_session=feature_store_session,
)

#### Set hyperparameters

In [ ]:
training_model.set_hyperparameters(objective="binary:logistic", num_round=50)

#### Specify training dataset 
Specify the training dataset created in the [Build Training Dataset](#Build-Training-Dataset) section.

In [ ]:
train_data = sagemaker.inputs.TrainingInput(
    dataset_uri_prefix,
    distribution="FullyReplicated",
    content_type="text/csv",
    s3_data_type="S3Prefix",
)
data_channels = {"train": train_data}

#### Start training

In [ ]:
training_model.fit(inputs=data_channels, logs=True)

## Set up Hosting for the Model

Once the training is done, we can deploy the trained model as an Amazon SageMaker real-time hosted endpoint. This will allow us to make predictions (or inference) from the model. Note that we don't have to host on the same instance (or type of instance) that we used to train. The endpoint deployment can be accomplished as follows. This takes 8-10 minutes to complete.

In [ ]:
predictor = training_model.deploy(initial_instance_count=1, instance_type="ml.m5.xlarge")

## SageMaker FeatureStore During Inference

SageMaker FeatureStore can be useful in supplementing data for inference requests because of the low-latency GetRecord functionality. For this demo, we will be given a TransactionId and query our online FeatureGroups for data on the transaction to build our inference request. 


In [ ]:
# Incoming inference request.
transaction_id = str(3450774)


# Helper to parse the feature value from the record.
def get_feature_value(record, feature_name):
    return str(list(filter(lambda r: r["FeatureName"] == feature_name, record))[0]["ValueAsString"])


transaction_response = featurestore_runtime.get_record(
    FeatureGroupName=transaction_feature_group_name, RecordIdentifierValueAsString=transaction_id
)
transaction_record = transaction_response["Record"]

transaction_test_data = [
    get_feature_value(transaction_record, "TransactionDT"),
    get_feature_value(transaction_record, "TransactionAmt"),
    get_feature_value(transaction_record, "card1"),
    get_feature_value(transaction_record, "card2"),
    get_feature_value(transaction_record, "card3"),
    get_feature_value(transaction_record, "card5"),
    get_feature_value(transaction_record, "card_type_credit"),
    get_feature_value(transaction_record, "card_type_debit"),
    get_feature_value(transaction_record, "card_bank_american_express"),
    get_feature_value(transaction_record, "card_bank_discover"),
    get_feature_value(transaction_record, "card_bank_mastercard"),
    get_feature_value(transaction_record, "card_bank_visa"),
]

identity_response = featurestore_runtime.get_record(
    FeatureGroupName=identity_feature_group_name, RecordIdentifierValueAsString=transaction_id
)
identity_record = identity_response["Record"]
id_test_data = [
    get_feature_value(identity_record, "id_01"),
    get_feature_value(identity_record, "id_02"),
    get_feature_value(identity_record, "id_03"),
    get_feature_value(identity_record, "id_04"),
    get_feature_value(identity_record, "id_05"),
]

# Join all pieces for inference request.
inference_request = []
inference_request.extend(transaction_test_data[:])
inference_request.extend(id_test_data[:])

inference_request

In [ ]:
import json

results = predictor.predict(",".join(inference_request), initial_args={"ContentType": "text/csv"})
prediction = json.loads(results)
print(prediction)

## Cleanup Resources

In [ ]:
predictor.delete_endpoint()

In [ ]:
identity_feature_group.delete()
transaction_feature_group.delete()

In [ ]:
# restore original boto3 version
%pip install 'boto3=={}'.format(original_boto3_version)

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/us-east-1/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/us-east-2/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/us-west-1/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/ca-central-1/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/sa-east-1/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/eu-west-1/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/eu-west-2/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/eu-west-3/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/eu-central-1/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/eu-north-1/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/ap-southeast-1/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/ap-southeast-2/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/ap-northeast-1/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/ap-northeast-2/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://h75twx4l60.execute-api.us-west-2.amazonaws.com/sagemaker-nb/ap-south-1/sagemaker-featurestore|sagemaker_featurestore_fraud_detection_python_sdk.ipynb)
